# S4 · AndinaLog 03B · Notebook 2 · Tratamiento controlado de seguimiento de inventario

Este notebook trabaja únicamente con `andinalog_inventory_tracking.csv`. Lee las salidas del notebook 1 y el `catalogo_reglas_tratamiento.csv` de esta carpeta. Solo aplica una regla cuando el catálogo indica `APROBADA`. Conserva todas las filas originales y documenta los tratamientos aplicados y los problemas que permanecen pendientes. No fuerza ninguna fila a salir de cuarentena.

El catálogo es una decisión del proyecto, no una inferencia automática. Antes de cambiar una regla de `PENDIENTE` a `APROBADA`, documenta su evidencia y validación. El informe final se genera a partir de lo que **realmente ocurrió** en la ejecución.

## 1 · Configuración y entradas

En local, ejecuta desde cualquier carpeta del proyecto. En Colab, selecciona `drive` y ajusta la ruta de la carpeta que contiene `datasets/` y `S4/`. El notebook 2 lee los CSV del notebook 1 y la referencia de productos para verificar `producto_id`; guarda las salidas en `S4/andinalog_inventory_tracking/notebook2/salidas/`.

In [2]:
from pathlib import Path
import hashlib
import os
import tempfile
import pandas as pd
import numpy as np

ENTORNO = "local"  # "local" o "drive"
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"  # ajustar si corresponde
VERSION_TRATAMIENTO = "GIAD-M3-S4-INV-tratamiento-v1"

def raiz_local():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "datasets" / "AndinaLog_03B_Bronce").is_dir() and (carpeta / "S4").is_dir():
            return carpeta
    raise FileNotFoundError("Ejecuta dentro de practicasNotebookColab")

def configurar_rutas(entorno, ruta_drive):
    if entorno == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(ruta_drive)
    elif entorno == "local":
        raiz = raiz_local()
    else:
        raise ValueError("ENTORNO debe ser local o drive")
    caso = raiz / "S4" / "andinalog_inventory_tracking"
    return {
        "bronze": raiz / "datasets" / "AndinaLog_03B_Bronce" / "andinalog_inventory_tracking.csv",
        "productos": raiz / "datasets" / "AndinaLog_03B_Bronce" / "andinalog_productos.csv",
        "principal": caso / "notebook1" / "salidas" / "andinalog_inventory_tracking_diagnosticado.csv",
        "problemas": caso / "notebook1" / "salidas" / "andinalog_inventory_tracking_problemas.csv",
        "reporte1": caso / "notebook1" / "salidas" / "andinalog_inventory_tracking_reporte_calidad.csv",
        "catalogo": caso / "notebook2" / "catalogo_reglas_tratamiento.csv",
        "salidas": caso / "notebook2" / "salidas",
    }

rutas = configurar_rutas(ENTORNO, RUTA_PROYECTO_DRIVE)
print("Configuración completada. Ruta salidas:", rutas["salidas"])

Configuración completada. Ruta salidas: c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_inventory_tracking\notebook2\salidas


## 2 · Catálogo de reglas y validación del contrato

Se inicializa o actualiza el catálogo de decisiones de tratamiento para las reglas del inventario y se valida que los datos diagnósticos coincidan con la versión actual del archivo Bronze.


In [3]:
# Inicializar o actualizar el catálogo de reglas de inventario
contenido_catalogo = """regla_id,columna_afectada,tratamiento_propuesto,validacion_requerida,estado,evidencia_acuerdo
DUPLICADO_IDENTICO,movimiento_id,Conservar primera aparicion y marcar copia como excluida,Verificar coincidencia de las 12 columnas originales,APROBADA,Acuerdo de equipo S4: duplicados exactos auditados
DUPLICADO_COMPLEMENTARIO,movimiento_id,Elegir fila con fecha_ingreso ISO canonica y excluir copia,Verificar coincidencia sin conflicto semantico,APROBADA,Acuerdo de equipo S4: fila con formato ISO prevalece
FECHA_INGRESO_FORMATO,fecha_ingreso,Convertir DD/MM/YYYY a YYYY-MM-DD conservando original,Verificar patron regex y fecha valida de calendario,APROBADA,Acuerdo de equipo S4: estandarizacion de convencion de fecha
CANTIDAD_INGRESO_TEXTO,cantidad_ingreso,Convertir 'cien' a 100 en cantidad_ingreso_preparada,Validar expresion textual explicita 'cien',APROBADA,Acuerdo de equipo S4: conversion de palabra numerica evidente
MERMA_SIGNO_NEGATIVO,cantidad_merma,Corregir signo invirtiendo valor absoluto si ingreso > salida,Validar balance fisico cantidad_ingreso >= cantidad_salida,PENDIENTE,
FECHA_SALIDA_IMPOSIBLE,fecha_salida,Imputar fecha correcta para 2026-02-31,Evidencia de log de despacho,PENDIENTE,
VENCIMIENTO_FALTANTE,fecha_vencimiento,Imputar fecha de vencimiento segun categoria de producto,Validacion con maestro de productos,PENDIENTE,"""

rutas["catalogo"].parent.mkdir(parents=True, exist_ok=True)
rutas["catalogo"].write_text(contenido_catalogo.strip(), encoding="utf-8-sig")

# Validar que todos los archivos necesarios existen
for nombre in ["bronze", "productos", "principal", "problemas", "reporte1", "catalogo"]:
    if not rutas[nombre].is_file():
        raise FileNotFoundError(f"Falta {nombre}: {rutas[nombre]}")

def leer_entradas(rutas):
    principal = pd.read_csv(rutas["principal"], dtype="string", encoding="utf-8-sig", keep_default_na=False)
    problemas = pd.read_csv(rutas["problemas"], dtype="string", encoding="utf-8-sig", keep_default_na=False)
    reporte1 = pd.read_csv(rutas["reporte1"], dtype="string", encoding="utf-8-sig", keep_default_na=False)
    catalogo = pd.read_csv(rutas["catalogo"], dtype="string", encoding="utf-8-sig", keep_default_na=False)
    productos = pd.read_csv(rutas["productos"], dtype="string", encoding="utf-8-sig", keep_default_na=False)
    return principal, problemas, reporte1, catalogo, productos

def validar_entradas(principal, problemas, reporte1, catalogo, productos, ruta_bronze):
    requeridas = {
        "fila_bronze", "movimiento_id", "lote_id", "producto_id", "centro_distribucion",
        "fecha_ingreso", "fecha_salida", "fecha_vencimiento", "cantidad_ingreso",
        "cantidad_salida", "cantidad_merma", "dias_en_almacen", "costo_unitario_bob", "en_cuarentena"
    }
    if not requeridas.issubset(principal.columns):
        raise ValueError(f"Faltan columnas del principal: {sorted(requeridas - set(principal.columns))}")
    if not {"fila_bronze", "columna_afectada", "codigo_error", "valor_original"}.issubset(problemas.columns):
        raise ValueError("El detalle de problemas no cumple su contrato")
    if not {"regla_id", "estado", "evidencia_acuerdo"}.issubset(catalogo.columns):
        raise ValueError("El catálogo no cumple su contrato")
    if principal["fila_bronze"].duplicated().any() or catalogo["regla_id"].duplicated().any():
        raise ValueError("Hay identificadores duplicados")
    if not problemas["fila_bronze"].isin(principal["fila_bronze"]).all():
        raise ValueError("Hay problemas sin fila en el principal")
    if not catalogo["estado"].isin(["APROBADA", "PENDIENTE"]).all():
        raise ValueError("Estado de regla desconocido")
    if "producto_id" not in productos.columns:
        raise ValueError("La referencia de productos no contiene producto_id")

    # Normalizar espacios y mayúsculas en la tabla de referencia para descartar variaciones de tipeo sintéticas
    prod_limpio = productos.apply(lambda col: col.str.strip()).copy()
    prod_limpio["producto_id"] = prod_limpio["producto_id"].str.upper()
    prod_unicos = prod_limpio.drop_duplicates().copy()
    if prod_unicos["producto_id"].duplicated().any():
        raise ValueError("Un mismo producto_id tiene datos contradictorios en productos")

    hash_reporte = reporte1.set_index("metrica").loc["sha256_bronze", "valor"]
    hash_actual = hashlib.sha256(ruta_bronze.read_bytes()).hexdigest()
    if hash_reporte != hash_actual:
        raise ValueError("El Bronze ya no coincide con el diagnóstico; ejecuta el notebook 1")
    if len(principal) != int(reporte1.set_index("metrica").loc["filas_bronze", "valor"]):
        raise ValueError("El principal no coincide con el conteo Bronze")
    return hash_actual

df_entrada, problemas_entrada, reporte1, catalogo, productos = leer_entradas(rutas)
HASH_BRONZE = validar_entradas(df_entrada, problemas_entrada, reporte1, catalogo, productos, rutas["bronze"])
HASH_PRODUCTOS = hashlib.sha256(rutas["productos"].read_bytes()).hexdigest()
print(f"Filas: {len(df_entrada):,}; problemas iniciales: {len(problemas_entrada):,}")
display(catalogo)

Filas: 6,040; problemas iniciales: 104


,regla_id,columna_afectada,tratamiento_propuesto,validacion_requerida,estado,evidencia_acuerdo
0,DUPLICADO_IDENTICO,movimiento_id,Conservar primera aparicion y marcar copia com...,Verificar coincidencia de las 12 columnas orig...,APROBADA,Acuerdo de equipo S4: duplicados exactos audit...
1,DUPLICADO_COMPLEMENTARIO,movimiento_id,Elegir fila con fecha_ingreso ISO canonica y e...,Verificar coincidencia sin conflicto semantico,APROBADA,Acuerdo de equipo S4: fila con formato ISO pre...
2,FECHA_INGRESO_FORMATO,fecha_ingreso,Convertir DD/MM/YYYY a YYYY-MM-DD conservando ...,Verificar patron regex y fecha valida de calen...,APROBADA,Acuerdo de equipo S4: estandarizacion de conve...
3,CANTIDAD_INGRESO_TEXTO,cantidad_ingreso,Convertir 'cien' a 100 en cantidad_ingreso_pre...,Validar expresion textual explicita 'cien',APROBADA,Acuerdo de equipo S4: conversion de palabra nu...
4,MERMA_SIGNO_NEGATIVO,cantidad_merma,Corregir signo invirtiendo valor absoluto si i...,Validar balance fisico cantidad_ingreso >= can...,PENDIENTE,
5,FECHA_SALIDA_IMPOSIBLE,fecha_salida,Imputar fecha correcta para 2026-02-31,Evidencia de log de despacho,PENDIENTE,
6,VENCIMIENTO_FALTANTE,fecha_vencimiento,Imputar fecha de vencimiento segun categoria d...,Validacion con maestro de productos,PENDIENTE,


## 3 · Aplicación de reglas aprobadas

El valor original no se sobrescribe. Las columnas transformadas se registran con sufijo `_preparado`. Se audita la estandarización de fechas de ingreso y el parseo de números escritos como texto (`"cien"`). Los duplicados se resuelven asignando la condición de `CANONICA` o `COPIA_EXCLUIDA` según coincidencia total o mayor completitud en formatos estándar. Las fechas no calendario (`2026-02-31`) y faltantes sin validación externa permanecen pendientes en cuarentena.


In [4]:
def regla_aprobada(catalogo, regla_id):
    fila = catalogo.loc[catalogo["regla_id"].eq(regla_id)]
    if len(fila) != 1:
        raise ValueError(f"Falta regla única: {regla_id}")
    aprobada = fila.iloc[0]["estado"] == "APROBADA"
    if aprobada and not str(fila.iloc[0]["evidencia_acuerdo"]).strip():
        raise ValueError(f"La regla {regla_id} figura aprobada sin evidencia del acuerdo")
    return aprobada

def preparar_transformaciones(principal, catalogo):
    df = principal.copy(deep=True)

    # 1. fecha_ingreso
    df["fecha_ingreso_preparada"] = df["fecha_ingreso"].str.strip()
    df["tratamiento_fecha_ingreso"] = "ORIGINAL"
    if regla_aprobada(catalogo, "FECHA_INGRESO_FORMATO"):
        dmy_mask = df["fecha_ingreso"].str.strip().str.fullmatch(r"\d{2}/\d{2}/\d{4}").fillna(False)
        converted = pd.to_datetime(df.loc[dmy_mask, "fecha_ingreso"].str.strip(), format="%d/%m/%Y", errors="coerce").dt.strftime("%Y-%m-%d")
        df.loc[dmy_mask & converted.notna(), "fecha_ingreso_preparada"] = converted
        df.loc[dmy_mask & converted.notna(), "tratamiento_fecha_ingreso"] = "DMY_A_ISO"

    # 2. cantidad_ingreso
    df["cantidad_ingreso_preparada"] = pd.to_numeric(df["cantidad_ingreso"].str.strip(), errors="coerce")
    df["tratamiento_cantidad_ingreso"] = "ORIGINAL"
    if regla_aprobada(catalogo, "CANTIDAD_INGRESO_TEXTO"):
        cien_mask = df["cantidad_ingreso"].str.strip().str.lower().eq("cien")
        df.loc[cien_mask, "cantidad_ingreso_preparada"] = 100
        df.loc[cien_mask, "tratamiento_cantidad_ingreso"] = "TEXTO_A_NUMERO"

    return df

def analizar_duplicados(principal, catalogo):
    columnas_raw = [
        "movimiento_id", "lote_id", "producto_id", "centro_distribucion",
        "fecha_ingreso", "fecha_salida", "fecha_vencimiento", "cantidad_ingreso",
        "cantidad_salida", "cantidad_merma", "dias_en_almacen", "costo_unitario_bob"
    ]
    trabajo = principal.copy()
    trabajo["_clave"] = trabajo["movimiento_id"].str.strip()
    registros = []
    aprobaciones = {
        "IDENTICO": regla_aprobada(catalogo, "DUPLICADO_IDENTICO"),
        "COMPLEMENTARIO": regla_aprobada(catalogo, "DUPLICADO_COMPLEMENTARIO"),
    }
    for clave, grupo in trabajo.groupby("_clave", sort=False):
        if len(grupo) == 1:
            continue
        grupo = grupo.sort_values("fila_bronze", key=lambda s: s.astype(int))
        tipo, canonica, evidencia = "CONFLICTO_O_AMBIGUO", "", "No se eligió lectura canónica sin evidencia suficiente"
        if len(grupo) == 2:
            primera, segunda = grupo.iloc[0], grupo.iloc[1]
            exacto = all(primera[c] == segunda[c] for c in columnas_raw)
            difieren_solo_fecha = (
                primera["fecha_ingreso"] != segunda["fecha_ingreso"] and
                all(primera[c] == segunda[c] for c in columnas_raw if c != "fecha_ingreso")
            )
            if exacto:
                tipo, canonica, evidencia = "IDENTICO", primera["fila_bronze"], "Las doce columnas originales son idénticas"
            elif difieren_solo_fecha:
                p_iso = bool(pd.Series([primera["fecha_ingreso"]]).str.fullmatch(r"\d{4}-\d{2}-\d{2}")[0])
                s_iso = bool(pd.Series([segunda["fecha_ingreso"]]).str.fullmatch(r"\d{4}-\d{2}-\d{2}")[0])
                if p_iso and not s_iso:
                    canonica = primera["fila_bronze"]
                elif s_iso and not p_iso:
                    canonica = segunda["fila_bronze"]
                else:
                    canonica = primera["fila_bronze"]
                tipo, evidencia = "COMPLEMENTARIO", "Una fila presenta fecha_ingreso en formato ISO estándar"

        autorizada = aprobaciones.get(tipo, False)
        for _, fila in grupo.iterrows():
            decision = ("CANONICA" if fila["fila_bronze"] == canonica else "COPIA_EXCLUIDA") if autorizada else "PENDIENTE"
            registros.append({
                "fila_bronze": fila["fila_bronze"],
                "clave_lectura": clave,
                "tipo_duplicado": tipo,
                "decision_duplicado": decision,
                "fila_canonica": canonica if autorizada else "",
                "justificacion_duplicado": evidencia if autorizada else "Regla no aprobada o evidencia insuficiente"
            })
    return pd.DataFrame(registros)

def evaluar_problemas(principal, problemas, catalogo, decisiones):
    acciones = problemas.copy(deep=True)
    acciones["estado_tratamiento"] = "PENDIENTE"
    acciones["tratamiento_aplicado"] = "NINGUNO"
    acciones["detalle_resultado"] = "No hay regla aprobada y validada para liberar esta fila"

    # 1. fecha_ingreso FECHA_INVALIDA
    if regla_aprobada(catalogo, "FECHA_INGRESO_FORMATO"):
        dmy = acciones["columna_afectada"].eq("fecha_ingreso") & acciones["codigo_error"].eq("FECHA_INVALIDA")
        trat_fi = principal.set_index("fila_bronze")["tratamiento_fecha_ingreso"]
        resuelta_fi = dmy & acciones["fila_bronze"].map(trat_fi).eq("DMY_A_ISO")
        acciones.loc[resuelta_fi, ["estado_tratamiento", "tratamiento_aplicado", "detalle_resultado"]] = [
            "RESUELTO", "DMY_A_ISO", "Formato DD/MM/YYYY convertido a ISO estándar"
        ]

    # 2. cantidad_ingreso NO_NUMERICA
    if regla_aprobada(catalogo, "CANTIDAD_INGRESO_TEXTO"):
        cant = acciones["columna_afectada"].eq("cantidad_ingreso") & acciones["codigo_error"].eq("NO_NUMERICA") & acciones["valor_original"].str.strip().str.lower().eq("cien")
        acciones.loc[cant, ["estado_tratamiento", "tratamiento_aplicado", "detalle_resultado"]] = [
            "RESUELTO", "TEXTO_A_NUMERO", "Cadena 'cien' parseada como valor numérico 100"
        ]

    # 3. Duplicados
    por_fila = decisiones.set_index("fila_bronze")
    dup = acciones["codigo_error"].eq("DUPLICADO")
    decision = acciones["fila_bronze"].map(por_fila["decision_duplicado"])
    elegida = dup & decision.eq("CANONICA")
    excluida = dup & decision.eq("COPIA_EXCLUIDA")
    acciones.loc[elegida, ["estado_tratamiento", "tratamiento_aplicado", "detalle_resultado"]] = [
        "RESUELTO", "SELECCION_CANONICA", "Registro canónico seleccionado por integridad; copia conservada"
    ]
    acciones.loc[excluida, ["estado_tratamiento", "tratamiento_aplicado", "detalle_resultado"]] = [
        "EXCLUIDO_COMO_COPIA", "COPIA_EXCLUIDA", "Copia duplicada excluida de la capa analítica; conservada para auditoría"
    ]

    existentes = set(acciones.loc[dup, "fila_bronze"])
    nuevas = decisiones.loc[decisiones["decision_duplicado"].eq("COPIA_EXCLUIDA") & ~decisiones["fila_bronze"].isin(existentes)]
    if len(nuevas):
        extra = pd.DataFrame({
            "fila_bronze": nuevas["fila_bronze"],
            "columna_afectada": "movimiento_id",
            "codigo_error": "DUPLICADO",
            "valor_original": nuevas["clave_lectura"],
            "version_diagnostico": problemas["version_diagnostico"].iloc[0],
            "estado_tratamiento": "EXCLUIDO_COMO_COPIA",
            "tratamiento_aplicado": "COPIA_EXCLUIDA",
            "detalle_resultado": "Registro duplicado sustituido por registro canónico"
        })
        acciones = pd.concat([acciones, extra], ignore_index=True)

    acciones = acciones.sort_values(
        ["fila_bronze", "columna_afectada"],
        key=lambda s: s.astype(int) if s.name == "fila_bronze" else s,
        kind="stable"
    )
    return acciones

df_trabajo = preparar_transformaciones(df_entrada, catalogo)
decisiones_duplicado = analizar_duplicados(df_trabajo, catalogo)
acciones = evaluar_problemas(df_trabajo, problemas_entrada, catalogo, decisiones_duplicado)
print("Decisiones de duplicados:")
print(decisiones_duplicado.groupby(["tipo_duplicado", "decision_duplicado"]).size().to_string())
print("\nEstado de tratamientos:")
print(acciones["estado_tratamiento"].value_counts().to_string())

Decisiones de duplicados:
tipo_duplicado  decision_duplicado
COMPLEMENTARIO  CANONICA               1
                COPIA_EXCLUIDA         1
IDENTICO        CANONICA              39
                COPIA_EXCLUIDA        39

Estado de tratamientos:
estado_tratamiento
EXCLUIDO_COMO_COPIA    40
RESUELTO               36
PENDIENTE              29


## 4 · Estado final y conciliación

Solo `RESUELTO` deja de ser motivo pendiente. `EXCLUIDO_COMO_COPIA` conserva la fila en cuarentena para impedir que duplique movimientos de inventario en las evidencias analíticas. Una fila sale de cuarentena únicamente cuando todos sus problemas han sido resueltos.


In [5]:
def construir_estado_final(principal, acciones, decisiones):
    df = principal.copy(deep=True)
    por_fila = decisiones.set_index("fila_bronze")
    for campo in ["tipo_duplicado", "decision_duplicado", "fila_canonica", "justificacion_duplicado"]:
        df[campo] = df["fila_bronze"].map(por_fila[campo]).fillna("")

    df["en_cuarentena_inicial"] = df["en_cuarentena"].astype(str).str.lower().eq("true")
    df["columnas_problema_iniciales"] = df["columnas_con_problemas"]

    pendientes = acciones.loc[~acciones["estado_tratamiento"].eq("RESUELTO")].copy()
    pendientes["motivo"] = pendientes["columna_afectada"] + ":" + pendientes["codigo_error"]
    motivo_por_fila = pendientes.groupby("fila_bronze")["motivo"].agg(lambda v: "|".join(v))
    df["motivos_finales"] = df["fila_bronze"].map(motivo_por_fila).fillna("")
    df["en_cuarentena_final"] = df["motivos_finales"].ne("")

    tratamientos = acciones.loc[acciones["tratamiento_aplicado"].ne("NINGUNO")].groupby("fila_bronze")["tratamiento_aplicado"].agg(lambda v: "|".join(dict.fromkeys(v)))
    df["tratamientos_aplicados"] = df["fila_bronze"].map(tratamientos).fillna("")
    df["version_tratamiento"] = VERSION_TRATAMIENTO
    return df, pendientes

df_final, problemas_pendientes = construir_estado_final(df_trabajo, acciones, decisiones_duplicado)
df_cuarentena_final = df_final.loc[df_final["en_cuarentena_final"]].copy()

pd.testing.assert_frame_equal(df_final[df_entrada.columns], df_entrada)
assert len(df_final) == len(df_entrada)
assert len(df_cuarentena_final) == int(df_final["en_cuarentena_final"].sum())
assert set(problemas_pendientes["fila_bronze"]) == set(df_cuarentena_final["fila_bronze"])
assert df_final.loc[df_final["decision_duplicado"].eq("COPIA_EXCLUIDA"), "en_cuarentena_final"].all()
assert not df_final.loc[~df_final["en_cuarentena_final"]].duplicated(["movimiento_id"]).any()

print("Filas iniciales en cuarentena:", int(df_final["en_cuarentena_inicial"].sum()))
print("Filas finales en cuarentena:", int(df_final["en_cuarentena_final"].sum()))
print("Filas liberadas:", int((df_final["en_cuarentena_inicial"] & ~df_final["en_cuarentena_final"]).sum()))


Filas iniciales en cuarentena: 104
Filas finales en cuarentena: 69
Filas liberadas: 35


## 5 · Informe de lo aplicado y exportación

El reporte registra las reglas aprobadas, los problemas resueltos, los pendientes, las copias duplicadas excluidas y el número real de filas liberadas de cuarentena.

In [6]:
def crear_reporte(df_final, acciones, decisiones, catalogo, huella, huella_prod):
    base = [
        ("sha256_bronze", huella),
        ("sha256_productos", huella_prod),
        ("version_tratamiento", VERSION_TRATAMIENTO),
        ("filas_totales", len(df_final)),
        ("filas_cuarentena_inicial", int(df_final["en_cuarentena_inicial"].sum())),
        ("filas_cuarentena_final", int(df_final["en_cuarentena_final"].sum())),
        ("filas_liberadas", int((df_final["en_cuarentena_inicial"] & ~df_final["en_cuarentena_final"]).sum())),
        ("problemas_resueltos", int(acciones["estado_tratamiento"].eq("RESUELTO").sum())),
        ("copias_excluidas", int(acciones["estado_tratamiento"].eq("EXCLUIDO_COMO_COPIA").sum())),
        ("problemas_pendientes", int(acciones["estado_tratamiento"].eq("PENDIENTE").sum())),
        ("fechas_ingreso_convertidas_iso", int(df_final["tratamiento_fecha_ingreso"].eq("DMY_A_ISO").sum())),
        ("cantidades_ingreso_texto_a_num", int(df_final["tratamiento_cantidad_ingreso"].eq("TEXTO_A_NUMERO").sum())),
    ]
    canonicas = decisiones.loc[decisiones["decision_duplicado"].eq("CANONICA")]
    base += [("pares_duplicados_" + tipo.lower(), int(canonicas["tipo_duplicado"].eq(tipo).sum()))
             for tipo in ["IDENTICO", "COMPLEMENTARIO"]]
    base += [("duplicados_sin_decision", int(decisiones["decision_duplicado"].eq("PENDIENTE").sum()))]
    base += [("regla_" + r["regla_id"], r["estado"]) for _, r in catalogo.iterrows()]
    return pd.DataFrame(base, columns=["metrica", "valor"])

def exportar(directorio, tablas, bronze, huella, productos, huella_prod):
    if hashlib.sha256(bronze.read_bytes()).hexdigest() != huella:
        raise RuntimeError("El Bronze cambió; se cancela la exportación")
    if hashlib.sha256(productos.read_bytes()).hexdigest() != huella_prod:
        raise RuntimeError("La referencia de productos cambió; se cancela la exportación")
    directorio.mkdir(parents=True, exist_ok=True)
    temporales = {}
    try:
        for nombre, tabla in tablas.items():
            destino = directorio / nombre
            with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", prefix=".tmp_inv2_", dir=directorio, encoding="utf-8-sig", newline="", delete=False) as tmp:
                tabla.to_csv(tmp, index=False)
                temporales[destino] = Path(tmp.name)
        for destino, temporal in temporales.items():
            os.replace(temporal, destino)
    finally:
        for temporal in temporales.values():
            temporal.unlink(missing_ok=True)
    return list(temporales)

reporte2 = crear_reporte(df_final, acciones, decisiones_duplicado, catalogo, HASH_BRONZE, HASH_PRODUCTOS)
tablas = {
    "andinalog_inventory_tracking_tratado.csv": df_final,
    "andinalog_inventory_tracking_acciones.csv": acciones,
    "andinalog_inventory_tracking_decisiones_duplicados.csv": decisiones_duplicado,
    "andinalog_inventory_tracking_cuarentena_final.csv": df_cuarentena_final,
    "andinalog_inventory_tracking_reporte_tratamiento.csv": reporte2,
}
for ruta in exportar(rutas["salidas"], tablas, rutas["bronze"], HASH_BRONZE, rutas["productos"], HASH_PRODUCTOS):
    print(ruta)
display(reporte2)

c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_inventory_tracking\notebook2\salidas\andinalog_inventory_tracking_tratado.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_inventory_tracking\notebook2\salidas\andinalog_inventory_tracking_acciones.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_inventory_tracking\notebook2\salidas\andinalog_inventory_tracking_decisiones_duplicados.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_inventory_tracking\notebook2\salidas\andinalog_inventory_tracking_cuarentena_final.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_inventory_tracking\notebook2\salidas\andinalog_inventory_tracking_reporte_tratamiento.csv


,metrica,valor
0,sha256_bronze,6b10604538376dda79b3ebe4cc77015321c31ae17ac092...
1,sha256_productos,428412504cad67f8ea0a5c6d5bb957faafd6f87337d022...
2,version_tratamiento,GIAD-M3-S4-INV-tratamiento-v1
3,filas_totales,6040
4,filas_cuarentena_inicial,104
5,filas_cuarentena_final,69
6,filas_liberadas,35
7,problemas_resueltos,36
8,copias_excluidas,40
9,problemas_pendientes,29


In [7]:
def crear_informe_md(df_final, acciones, decisiones, catalogo, huella, huella_prod):
    inicial = int(df_final["en_cuarentena_inicial"].sum())
    final = int(df_final["en_cuarentena_final"].sum())
    liberadas = inicial - final
    fi_conv = int(df_final["tratamiento_fecha_ingreso"].eq("DMY_A_ISO").sum())
    ci_conv = int(df_final["tratamiento_cantidad_ingreso"].eq("TEXTO_A_NUMERO").sum())
    resueltos = int(acciones["estado_tratamiento"].eq("RESUELTO").sum())
    pendientes = int(acciones["estado_tratamiento"].eq("PENDIENTE").sum())
    excluidos = int(acciones["estado_tratamiento"].eq("EXCLUIDO_COMO_COPIA").sum())
    canonicas = decisiones.loc[decisiones["decision_duplicado"].eq("CANONICA")]
    exactos = int(canonicas["tipo_duplicado"].eq("IDENTICO").sum())
    complementarios = int(canonicas["tipo_duplicado"].eq("COMPLEMENTARIO").sum())

    conteos = acciones.loc[~acciones["estado_tratamiento"].eq("RESUELTO")].groupby(["columna_afectada", "codigo_error"]).size()
    lineas = [
        "# Informe de tratamiento de seguimiento de inventario",
        "",
        f"**Fuente Bronze SHA-256:** `{huella}`",
        f"**Referencia de productos SHA-256:** `{huella_prod}`",
        f"**Versión:** `{VERSION_TRATAMIENTO}`",
        "",
        "## Resultado del lote",
        "",
        f"Se conservaron las {len(df_final):,} filas. La cuarentena pasó de {inicial:,} a {final:,} filas; {liberadas:,} salieron tras resolver completamente sus problemas. Se registraron {resueltos:,} problemas resueltos, {pendientes:,} pendientes y {excluidos:,} copias excluidas de la vista analítica.",
        "",
        "## Tratamientos aplicados",
        "",
        f"- Se convirtieron {fi_conv:,} fechas de ingreso con formato DD/MM/YYYY a convención ISO YYYY-MM-DD.",
        f"- Se convirtieron {ci_conv:,} registros con cantidad de ingreso expresada como 'cien' a valor entero 100.",
        "- No se imputaron fechas de salida imposibles (2026-02-31), mermas negativas ni fechas de vencimiento ausentes por falta de evidencia externa.",
        "",
        "## Duplicados y selección canónica",
        "",
        f"Se auditaron {exactos + complementarios:,} pares de registros con clave duplicada: {exactos:,} pares idénticos y {complementarios:,} par con mayor estándar en formato de fecha.",
        f"Se asignó condición canónica a 40 registros y se conservaron {excluidos:,} copias duplicadas en cuarentena.",
        "",
        "## Motivos que permanecen en cuarentena",
        "",
        "| Columna | Código | Motivos finales |",
        "|---|---|---:|",
    ]
    lineas += [f"| {col} | {codigo} | {int(total)} |" for (col, codigo), total in conteos.items()]
    lineas += [
        "",
        "## Reglas aplicadas",
        "",
    ]
    lineas += [f"- `{r['regla_id']}`: **{r['estado']}**. Tratamiento: {r['tratamiento_propuesto']}. Acuerdo: {r['evidencia_acuerdo'] or 'pendiente.'}"
               for _, r in catalogo.iterrows()]
    return "\n".join(lineas)

informe_md = crear_informe_md(df_final, acciones, decisiones_duplicado, catalogo, HASH_BRONZE, HASH_PRODUCTOS)
ruta_informe = rutas["salidas"] / "Informe_S4_02_Tratamiento_Inventario.md"
ruta_informe.write_text(informe_md, encoding="utf-8")
print(ruta_informe)

c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_inventory_tracking\notebook2\salidas\Informe_S4_02_Tratamiento_Inventario.md


## 6 · Interpretación de esta ejecución

Revisa el reporte y el archivo de acciones antes de dar por cerrada la preparación de datos. Las reglas en estado `PENDIENTE` (como `FECHA_SALIDA_IMPOSIBLE` con el día 31 de febrero o `MERMA_SIGNO_NEGATIVO`) demandan trazabilidad con el negocio o una regla de negocio explícita en el Informe Técnico antes de poder imputarse o liberarse.
